In [ ]:
import re
import numpy as np
import pandas as pd
from pathlib import Path

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"

BRAND_PART_PATH = DATA_DIR / "brand_analysis_part_enhanced.csv"
CLUSTER_DEF_PATH = DATA_DIR / "brand_tone_cluster.csv"             # 클러스터 정의표(이름/position/cluster_id)
TONE_CENTROID_CSV = DATA_DIR / "tone_centroid_embeddings.csv"      # ★ 이걸로 centroid↔이름 매핑 복구
TONE_CENTROID_NPY = DATA_DIR / "tone_centroid_embeddings.npy"

OUT_BY_BRAND = DATA_DIR / "brand_tone_cluster_by_brand.csv"

for p in [BRAND_PART_PATH, CLUSTER_DEF_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"없음: {p}")

# ----------------------------
# 0) 로드
# ----------------------------
brand_part_df = pd.read_csv(BRAND_PART_PATH)
cluster_def_df = pd.read_csv(CLUSTER_DEF_PATH)

print("[INFO] brand_part cols:", list(brand_part_df.columns))
print("[INFO] cluster_def cols:", list(cluster_def_df.columns))

# ----------------------------
# 1) centroid 로드 (CSV가 있으면 CSV 우선: tone_id(=클러스터명) 포함이라 매핑이 안전함)
# ----------------------------
if TONE_CENTROID_CSV.exists():
    cen_df = pd.read_csv(TONE_CENTROID_CSV)
    if "tone_id" not in cen_df.columns:
        raise ValueError("tone_centroid_embeddings.csv에 tone_id 컬럼이 없음")
    cen_names = cen_df["tone_id"].astype(str).tolist()
    vec_cols = [c for c in cen_df.columns if c != "tone_id"]
    tone_centroids = cen_df[vec_cols].to_numpy(dtype=np.float32)
    print("[INFO] centroids from CSV:", tone_centroids.shape, "names:", len(cen_names))
elif TONE_CENTROID_NPY.exists():
    tone_centroids = np.load(TONE_CENTROID_NPY).astype(np.float32)
    # CSV가 없으면 이름 매핑을 못함 → 여기서 멈추는 게 안전
    raise FileNotFoundError("tone_centroid_embeddings.csv가 없어서 centroid 이름 매핑 불가 (CSV 생성 필요)")
else:
    raise FileNotFoundError("tone_centroid_embeddings.csv / .npy 둘 다 없음")

# ----------------------------
# 2) centroid 자체가 한 덩어리로 붕괴했는지 체크 (여기가 문제면 무조건 쏠림 발생)
# ----------------------------
def cosine_matrix(A, B):
    A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    B = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return A @ B.T

C = cosine_matrix(tone_centroids, tone_centroids)
off = C[~np.eye(C.shape[0], dtype=bool)]
print("[CHECK] centroid cosine off-diag: min=%.4f mean=%.4f max=%.4f" % (off.min(), off.mean(), off.max()))
# max가 0.98~1.00에 몰려있으면 centroid들이 거의 같은 벡터 → 쏠림이 정상적으로 발생함

# ----------------------------
# 3) brand vector 만들기 (part embedding_vector 파싱)
# ----------------------------
dim = tone_centroids.shape[1]

def norm_text(x):
    s = "" if pd.isna(x) else str(x)
    s = s.replace("\u00a0", " ")
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

brand_part_df["brand_norm"] = brand_part_df["brand"].map(norm_text)

def parse_embedding_vector(val, expected_dim=dim):
    s = "" if pd.isna(val) else str(val)
    nums = re.findall(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", s)
    if not nums:
        return np.zeros(expected_dim, dtype=np.float32)
    arr = np.array(nums, dtype=np.float32)
    if arr.size < expected_dim:
        arr = np.concatenate([arr, np.zeros(expected_dim - arr.size, dtype=np.float32)])
    elif arr.size > expected_dim:
        arr = arr[:expected_dim]
    return arr.astype(np.float32)

part_vecs = np.vstack([parse_embedding_vector(v) for v in brand_part_df["embedding_vector"]]).astype(np.float32)
zero_cnt = int((np.linalg.norm(part_vecs, axis=1) == 0).sum())
print("[INFO] part_vecs:", part_vecs.shape, "zero:", zero_cnt)

# brand mean vec
brand_part_df["row_idx"] = np.arange(len(brand_part_df))
groups = brand_part_df.groupby("brand_norm")["row_idx"].apply(list).to_dict()

brand_names, brand_vecs, part_counts = [], [], []
for b, idxs in groups.items():
    X = part_vecs[idxs]
    brand_names.append(b)
    brand_vecs.append(X.mean(axis=0))
    part_counts.append(len(idxs))

brand_vecs = np.vstack(brand_vecs).astype(np.float32)
print("[INFO] brand_vecs:", brand_vecs.shape, "brands:", len(brand_names))

# ----------------------------
# 4) brand -> centroid argmax 분포 확인 (쏠림 재현 체크)
# ----------------------------
S = cosine_matrix(brand_vecs, tone_centroids)  # (B, K)
best_k = S.argmax(axis=1).astype(int)
best_sim = S.max(axis=1).astype(float)

cnt = pd.Series(best_k).value_counts().sort_index()
print("[CHECK] argmax cluster distribution:")
print(cnt.to_string())

# 상위 3개 클러스터까지 같이 보기 (진짜로 전부 0만 높은지 확인)
top3 = np.argsort(-S, axis=1)[:, :3]
sample = pd.DataFrame({
    "brand": brand_names,
    "top1": [cen_names[i] for i in top3[:,0]],
    "top1_sim": S[np.arange(len(brand_names)), top3[:,0]],
    "top2": [cen_names[i] for i in top3[:,1]],
    "top2_sim": S[np.arange(len(brand_names)), top3[:,1]],
    "top3": [cen_names[i] for i in top3[:,2]],
    "top3_sim": S[np.arange(len(brand_names)), top3[:,2]],
}).sort_values("top1_sim", ascending=False)

print("[CHECK] top3 preview:")
display(sample.head(12))

# ----------------------------
# 5) 정상 매핑으로 brand_tone_cluster_by_brand.csv 생성
#    - 클러스터 정의표(cluster_def_df)의 'brand'는 "클러스터명"임
#    - cen_names(=tone_id)도 "클러스터명"이어야 함
# ----------------------------
cluster_def_df["cluster_name"] = cluster_def_df["brand"].map(norm_text)
cluster_def_df["brand_tone_cluster"] = cluster_def_df["brand_tone_cluster"].astype(int)

name_to_cluster_id = cluster_def_df.set_index("cluster_name")["brand_tone_cluster"].to_dict()
name_to_position   = cluster_def_df.set_index("cluster_name")["brand_position"].to_dict()

top1_name = [cen_names[i] for i in best_k]
out = pd.DataFrame({
    "brand": brand_names,
    "cluster_name": top1_name,
    "brand_tone_cluster": [name_to_cluster_id.get(n, -1) for n in top1_name],
    "brand_position": [name_to_position.get(n, "") for n in top1_name],
    "brand_part_count": part_counts,
    "sim_to_centroid": best_sim,
})

# 매핑 실패(-1) 있으면 여기서 바로 보임
fail = out[out["brand_tone_cluster"] < 0]
print("[INFO] mapping fail rows:", len(fail))
if len(fail) > 0:
    print("[WARN] unmapped cluster_name sample:", fail["cluster_name"].unique()[:10])

out = out.sort_values(["brand_tone_cluster", "sim_to_centroid"], ascending=[True, False]).reset_index(drop=True)
out.to_csv(OUT_BY_BRAND, index=False, encoding="utf-8-sig")

print("[DONE] saved:", OUT_BY_BRAND, "rows:", len(out))
display(out.head(15))

[INFO] tone_centroids: (12, 768)
[INFO] part_vecs: (79, 768) zero: 0
[INFO] brand_vecs: (30, 768) brands: 30
[DONE] saved: /Users/mac/Desktop/project/STUDY-DATA/third_week/12_27/data/brand_tone_cluster_by_brand.csv rows: 30


,brand,brand_tone_cluster,cluster_name,brand_position,brand_part_count,sim_to_centroid
0,바이탈뷰티,0,Sensory-Luxury,고기능/클리니컬 톤,2,0.032145
1,일리윤,0,Sensory-Luxury,고기능/클리니컬 톤,3,0.024507
2,라보에이치,0,Sensory-Luxury,고기능/클리니컬 톤,3,0.021585
3,에스트라,0,Sensory-Luxury,고기능/클리니컬 톤,3,0.009553
4,타타 하퍼,0,Sensory-Luxury,고기능/클리니컬 톤,2,0.009212
5,아이오페,0,Sensory-Luxury,고기능/클리니컬 톤,3,-0.009444
6,홀리추얼,0,Sensory-Luxury,고기능/클리니컬 톤,2,-0.011729
7,아윤채,0,Sensory-Luxury,고기능/클리니컬 톤,2,-0.025862
8,이니스프리,0,Sensory-Luxury,고기능/클리니컬 톤,3,-0.038840
9,에스쁘아,0,Sensory-Luxury,고기능/클리니컬 톤,3,-0.042233
